# API
#### O objetivo deste notebook é consumir os dados da API da Netflix, para que possam alimentar o painel contendo as análises descritivas

## 0. Importações

In [39]:
import http.client
import json
import pandas as pd
from pandas import json_normalize
import requests
import time

## 1. Conectando com a API

In [41]:
conn = http.client.HTTPSConnection("netflix54.p.rapidapi.com")
headers = {
    'x-rapidapi-key': "f5b1bc69e3msh0f118dbbad3f35ep145147jsn5bce821b493a",
    'x-rapidapi-host': "netflix54.p.rapidapi.com"
}
#url = 'https://netflix54.p.rapidapi.com'

## 2. Requisitando dados da API

### 2.1 Buscando os IDs dos títulos presentes em nossos CSVs

In [46]:
titulos_viewing = pd.read_csv('../dados-transformados/Viewing_en.csv')
#titulos_list = pd.read_csv('../dados-transformados/MyList.csv')

titulos_viewing[['Title_Name', 'Temporada', 'Episodio']] = titulos_viewing['Title'].str.split(': ',n=2, expand=True)
titulos_viewing.drop('Title', axis=1, inplace=True)
titulos_viewing["Title_Name"] = titulos_viewing["Title_Name"].str.replace(r"\(.*?\)", "", regex=True)

limite_hora = 1000
intervalo = 3600 / limite_hora
ids = []

for titulo in titulos_viewing["Title_Name"].drop_duplicates():
    url = f"https://netflix54.p.rapidapi.com/search/?query={titulo}&limit=1&lang=en"

    retorno_id = requests.get(url, headers=headers)
    dados_id = retorno_id.json()

    if dados_id.get("results"):
        id_encontrado = dados_id["titles"][0]["summary"]["id"]
        ids.append(id_encontrado)
        print(f"{titulo} → {id_encontrado}")
    else:
        print(f"{titulo} → não encontrado")

    time.sleep(intervalo)

Bridgerton → não encontrado
Dr. House → não encontrado
Season 4  → não encontrado
Season 3  → não encontrado
Never Again Again → não encontrado
Never Say Never Again_hook_04 → não encontrado
The Bricklayer_hook_02_16x9 → não encontrado
Undercover Agent → não encontrado
Undercover Agent  → não encontrado
The Enforcer_hook_05 → não encontrado
Can Love Be Translated? → não encontrado
Envious → não encontrado
Luther → não encontrado
Clip 2 → não encontrado
Clip 4 → não encontrado
Teaser → não encontrado


KeyboardInterrupt: 

### 2.2 Trabalhando com os IDs retornados para obter mais informações sobre os títulos

In [36]:
# buscar os ids dos títulos presentes nos nossos dados, serão armazenados na lista ids

dados = pd.DataFrame()

#api = "/season/episodes/?ids=80077209%2C80117715&offset=0&limit=25&lang=en"
#res = requests.get(url+api , headers=headers)

for id_ in ids:
    url = f"https://netflix54.p.rapidapi.com/season/episodes/?ids={id_}&offset=0&limit=25&lang=en"

    retorno = requests.get(url, headers=headers)
    dados = retorno.json()

    print(f"ID {id_} coletado")

    time.sleep(intervalo)

## 3. Tranformando retorno da API em um DataFrame

In [37]:
#data = res.json()
for d in dados:
    df = pd.concat([dados,pd.json_normalize(d['episodes'])], ignore_index=True)

df.head()

,episodeId,bookmarkPosition,displayRuntime,runtime,title,availability.isPlayable,availability.availabilityDate,availability.availabilityStartTime,availability.unplayableCause,contextualSynopsis.text,...,interestingMoment._342x192.webp.value.image_key,summary.type,summary.unifiedEntityId,summary.id,summary.isOriginal,summary.liveEvent.hasLiveEvent,summary.idx,summary.episode,summary.season,summary.isPlayable
0,80077368,1689,2976,2976,Chapter One: The Vanishing of Will Byers,True,July 15,1468591200000,None,"On his way home from a friend's house, young W...",...,MERCH_STILL|dfa3e890-c9b5-11f0-8f55-12e5528d65...,episode,Video:80077368,80077368,True,False,1,1,1,True
1,80077369,-1,3384,3384,Chapter Two: The Weirdo on Maple Street,True,July 15,1468591200000,None,"Lucas, Mike and Dustin try to talk to the girl...",...,MERCH_STILL|05c962c0-7ec5-11e6-8984-0ed4190bdb...,episode,Video:80077369,80077369,True,False,2,2,1,True
2,80077370,-1,3171,3171,"Chapter Three: Holly, Jolly",True,July 15,1468591200000,None,An increasingly concerned Nancy looks for Barb...,...,MERCH_STILL|2f2ec330-c9b6-11f0-a1bb-0e6a0bfa99...,episode,Video:80077370,80077370,True,False,3,3,1,True
3,80077371,-1,3085,3085,Chapter Four: The Body,True,July 15,1468591200000,None,"Refusing to believe Will is dead, Joyce tries ...",...,MERCH_STILL|089469a0-7ec5-11e6-9bbe-0e6b5a566d...,episode,Video:80077371,80077371,True,False,4,4,1,True
4,80077372,-1,3236,3236,Chapter Five: The Flea and the Acrobat,True,July 15,1468591200000,None,Hopper breaks into the lab while Nancy and Jon...,...,MERCH_STILL|0a1675c0-7ec5-11e6-b391-0a77d1264d...,episode,Video:80077372,80077372,True,False,5,5,1,True


# 4. Exportando informações da API para um arquivo .csv

In [38]:
dados.to_csv('../dados-transformados/api.csv', index=False)